![](https://www.soyhenry.com/_next/static/media/HenryLogo.bb57fd6f.svg)

# Cap 04 — Enrutamiento Condicional

profesor [Carlos Daniel Jiménez](danieljimenez88m@gmail.com)


Conectamos los 3 dominios con un router que elige el especialista correcto:
- `add_conditional_edges`: aristas que dependen del estado
- Función de routing: lee el estado y devuelve el nombre del siguiente nodo
- **Routing por keywords** (rápido, frágil)
- **Routing por LLM** con `with_structured_output` (robusto, más lento)

**Dominio**: Los 3 dominios (Nolan, King, Davis)  
**Flujo**: `START → router → [nolan|king|davis|general] → specialist → END`

In [ ]:
import os
import json
import time
from pathlib import Path
from typing import Literal
from dotenv import load_dotenv

load_dotenv()

from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END, MessagesState
from pydantic import BaseModel, Field
from rich import print as rprint

# Cargar todos los datos
data_dir = Path("../00_datos")
nolan_films = json.loads((data_dir / "nolan_films.json").read_text(encoding="utf-8"))
king_books = json.loads((data_dir / "king_books.json").read_text(encoding="utf-8"))
davis_albums = json.loads((data_dir / "davis_albums.json").read_text(encoding="utf-8"))

print(f"Nolan: {len(nolan_films)}, King: {len(king_books)}, Davis: {len(davis_albums)}")

## 1. Estado con campo `domain`

El estado necesita un campo `domain` que el router asignará y la función de routing leerá.

In [ ]:
from typing import Optional

class CulturalState(MessagesState):
    domain: Optional[str]
    ruta_tomada: str

# La función de routing es una función PURA que lee el estado
def cultural_route(state: CulturalState) -> Literal["nolan_specialist", "king_specialist", "davis_specialist", "general"]:
    """Lee state['domain'] y devuelve el nombre del siguiente nodo."""
    domain = state.get("domain", "general")
    routing_map = {
        "nolan": "nolan_specialist",
        "king": "king_specialist", 
        "davis": "davis_specialist",
    }
    return routing_map.get(domain, "general")

print("Función de routing definida ✓")

## 2. Routing por Keywords (rápido pero frágil)

In [ ]:
NOLAN_KEYWORDS = {"nolan", "inception", "memento", "interstellar", "dunkirk", "tenet", 
                  "oppenheimer", "prestige", "batman", "dark knight", "película", "film"}
KING_KEYWORDS = {"king", "stephen", "it", "shining", "carrie", "misery", "pennywise",
                 "derry", "horror", "libro", "novela", "terror"}
DAVIS_KEYWORDS = {"davis", "miles", "jazz", "kind of blue", "bitches brew", "bebop",
                  "fusion", "coltrane", "álbum", "música", "trompeta"}

def node_router_keywords(state: CulturalState) -> dict:
    """Router basado en keywords — O(n) pero sin latencia de LLM."""
    query = state["messages"][-1].content.lower()
    
    nolan_score = sum(1 for k in NOLAN_KEYWORDS if k in query)
    king_score = sum(1 for k in KING_KEYWORDS if k in query)
    davis_score = sum(1 for k in DAVIS_KEYWORDS if k in query)
    
    scores = {"nolan": nolan_score, "king": king_score, "davis": davis_score}
    best = max(scores, key=scores.get)
    domain = best if scores[best] > 0 else "general"
    
    print(f"  Scores: {scores} → {domain}")
    return {"domain": domain, "ruta_tomada": f"keyword:{domain}"}

print("Router de keywords definido ✓")

In [ ]:
llm = ChatOpenAI(model=os.getenv("OPENAI_MODEL", "gpt-5-mini"), temperature=0)

SYSTEM_NOLAN = "Eres experto en cine de Christopher Nolan. Responde en español de forma concisa."
SYSTEM_KING = "Eres experto en libros de Stephen King. Responde en español de forma concisa."
SYSTEM_DAVIS = "Eres experto en música de Miles Davis. Responde en español de forma concisa."

def make_specialist(system_prompt: str):
    def specialist(state: CulturalState) -> dict:
        response = llm.invoke([SystemMessage(content=system_prompt)] + state["messages"])
        return {"messages": [response]}
    return specialist

def node_general(state: CulturalState) -> dict:
    response = llm.invoke([
        SystemMessage(content="Eres un asistente cultural. Nuestros dominios son: películas de Nolan, libros de King, música de Davis."),
        *state["messages"]
    ])
    return {"messages": [response]}

# Construir grafo con keyword routing
builder = StateGraph(CulturalState)
builder.add_node("router", node_router_keywords)
builder.add_node("nolan_specialist", make_specialist(SYSTEM_NOLAN))
builder.add_node("king_specialist", make_specialist(SYSTEM_KING))
builder.add_node("davis_specialist", make_specialist(SYSTEM_DAVIS))
builder.add_node("general", node_general)

builder.add_edge(START, "router")
builder.add_conditional_edges("router", cultural_route)
for node in ["nolan_specialist", "king_specialist", "davis_specialist", "general"]:
    builder.add_edge(node, END)

graph_keywords = builder.compile()
print("Grafo con keyword routing compilado ✓")

In [ ]:
# Prueba del routing
queries = [
    "¿Cuál es la técnica narrativa de Inception?",
    "¿De qué trata It de Stephen King?",
    "¿Qué es Kind of Blue?",
    "¿Cuál es la capital de Francia?",  # general fallback
]

for query in queries:
    t0 = time.time()
    result = graph_keywords.invoke({"messages": [HumanMessage(content=query)], "domain": None, "ruta_tomada": ""})
    elapsed = time.time() - t0
    print(f"\nQuery: {query[:50]}")
    print(f"Ruta: {result['ruta_tomada']} | Tiempo: {elapsed:.2f}s")
    print(f"Respuesta: {result['messages'][-1].content[:150]}...")

## 3. Routing por LLM con `with_structured_output`

Más robusto: el LLM entiende el contexto y clasifica correctamente incluso sin keywords exactas.

In [ ]:
class DomainRoute(BaseModel):
    """Clasificación del dominio cultural."""
    domain: Literal["nolan", "king", "davis", "general"] = Field(
        description="El dominio cultural más relevante para la consulta del usuario"
    )
    confidence: float = Field(ge=0, le=1, description="Confianza en la clasificación")
    reasoning: str = Field(description="Razón breve de la clasificación")

structured_llm = llm.with_structured_output(DomainRoute)

ROUTER_SYSTEM = """Clasifica la consulta del usuario en uno de estos dominios:
- "nolan": películas de Christopher Nolan
- "king": libros de Stephen King  
- "davis": álbumes de Miles Davis
- "general": otras consultas"""

def node_router_llm(state: CulturalState) -> dict:
    """Router basado en LLM — más robusto pero con latencia extra."""
    result: DomainRoute = structured_llm.invoke(
        [SystemMessage(content=ROUTER_SYSTEM)] + state["messages"]
    )
    return {
        "domain": result.domain,
        "ruta_tomada": f"llm:{result.domain}(conf={result.confidence:.2f})"
    }

# Grafo con LLM routing
builder2 = StateGraph(CulturalState)
builder2.add_node("router", node_router_llm)
builder2.add_node("nolan_specialist", make_specialist(SYSTEM_NOLAN))
builder2.add_node("king_specialist", make_specialist(SYSTEM_KING))
builder2.add_node("davis_specialist", make_specialist(SYSTEM_DAVIS))
builder2.add_node("general", node_general)

builder2.add_edge(START, "router")
builder2.add_conditional_edges("router", cultural_route)
for node in ["nolan_specialist", "king_specialist", "davis_specialist", "general"]:
    builder2.add_edge(node, END)

graph_llm = builder2.compile()
print("Grafo con LLM routing compilado ✓")

In [ ]:
# Benchmark: keywords vs LLM routing
print("=== Benchmark: Keywords vs LLM Routing ===\n")

tricky_queries = [
    "El director de Dunkirk filmó Batman",       # nolan (sin keyword 'nolan')
    "El payaso de Derry asusta a los niños",     # king (sin keyword 'king')
    "El jazzista de Birth of the Cool",           # davis (sin keyword 'davis')
]

for query in tricky_queries:
    t0 = time.time()
    r_kw = graph_keywords.invoke({"messages": [HumanMessage(content=query)], "domain": None, "ruta_tomada": ""})
    t_kw = time.time() - t0
    
    t0 = time.time()
    r_llm = graph_llm.invoke({"messages": [HumanMessage(content=query)], "domain": None, "ruta_tomada": ""})
    t_llm = time.time() - t0
    
    print(f"Query: '{query}'")
    print(f"  Keywords: {r_kw['ruta_tomada']} ({t_kw:.2f}s)")
    print(f"  LLM:      {r_llm['ruta_tomada']} ({t_llm:.2f}s)")
    print()

## Resumen del Capítulo

| Concepto | Descripción |
|---------|-------------|
| `add_conditional_edges` | Añade aristas condicionales basadas en el estado |
| Función de routing | Función pura: `(state) → nombre_del_nodo` |
| Keyword routing | Rápido, frágil con frases indirectas |
| LLM routing | Robusto, latencia extra de ~0.5-1s |
| `with_structured_output` | Fuerza al LLM a devolver JSON estructurado |
| `ruta_tomada` | Campo de estado para auditar la ruta elegida |

**Decisión de diseño**: Usar LLM routing para precisión, keyword routing para bajo costo.

**Próximo capítulo**: Salida estructurada con Pydantic